In [ ]:
## Importing Libraries

import os
import cv2
import copy
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn

from torchvision import models
import torchvision.transforms as transforms

In [ ]:
## Setting GPU and Target Columns

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_HEIGHT = 256
IMAGE_WIDTH = 512

TARGET_COLUMNS = ["Dry Clover", "Dry Dead", "Dry Green", "Dry Total", "GDM"]

print("Using Device :", DEVICE)

In [ ]:
## Loading Paths

DATASET_PATH = r"Dataset"
MODEL_PATH = r"Models\model.pth"

IMAGE_PATH = os.path.join(DATASET_PATH, "train", "ID1011485656.jpg")

print("Paths Loaded Successfully.")

In [ ]:
## Transforming Image

transform = transforms.Compose([
    transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
    transforms.ToTensor(),

    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
## Setting up Model

weights = models.EfficientNet_B3_Weights.DEFAULT
model = models.efficientnet_b3(weights=weights)

in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(0.30),
    nn.Linear(in_features,512),
    nn.ReLU(),

    nn.Dropout(0.20),
    nn.Linear(512,128),
    nn.ReLU(),

    nn.Linear(128,5)
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print("Model Loaded Successfully.")

In [ ]:
## Loading Image

image = Image.open(IMAGE_PATH).convert("RGB")
display(image)

In [ ]:
## Resizing Image

rgb_image = image.resize((IMAGE_WIDTH, IMAGE_HEIGHT))
rgb_image = np.array(rgb_image).astype(np.float32) / 255.0

input_tensor = transform(image)
input_tensor = input_tensor.unsqueeze(0)
input_tensor = input_tensor.to(DEVICE)

print(input_tensor.shape)

In [ ]:
## Registering Hooks

activations = None
gradients = None


def forward_hook(module, input, output):
    global activations
    activations = output


def backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0]


target_layer = model.features[-1][0]

forward_handle = target_layer.register_forward_hook(forward_hook)
backward_handle = target_layer.register_full_backward_hook(backward_hook)

print("Hooks Registered Successfully.")

In [ ]:
## Forward Hook

output = model(input_tensor)
print("Predictions:\n")

for name, value in zip(TARGET_COLUMNS, output[0]):
    print(f"{name:15s} : {value.item():.2f}")

In [ ]:
## Selecting Target

TARGET_INDEX = 0
print("Selected Target :", TARGET_COLUMNS[TARGET_INDEX])

In [ ]:
## Computing Gradients

model.zero_grad()
output[0, TARGET_INDEX].backward()

print("Gradient Shape :", gradients.shape)
print("Activation Shape :", activations.shape)

In [ ]:
## Computing Channel Weights

weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
print(weights.shape)

In [ ]:
## Generating CAM

cam = torch.sum(weights * activations, dim=1)
cam = torch.relu(cam)

cam = cam.squeeze().detach().cpu().numpy()
print(cam.shape)


In [ ]:
## Normalizing CAM

cam = cam - cam.min()
cam = cam / (cam.max() + 1e-8)

print(cam.min(), cam.max())

In [ ]:
## Resizing Heatmap

heatmap = cv2.resize(cam, (IMAGE_WIDTH, IMAGE_HEIGHT))
print(heatmap.shape)

In [ ]:
## Colorizing Heatmap

heatmap_color = np.uint8(255 * heatmap)
heatmap_color = cv2.applyColorMap(heatmap_color, cv2.COLORMAP_JET)
heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

print(heatmap_color.shape)

In [ ]:
## Creatig Overlay

overlay = cv2.addWeighted(np.uint8(rgb_image*255), 0.6, heatmap_color, 0.4, 0)

In [ ]:
## Visualization

plt.figure(figsize=(18,6))

plt.subplot(1,3,1)
plt.imshow(rgb_image)
plt.title("Original")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(heatmap_color)
plt.title("Heatmap")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(overlay)
plt.title("Overlay")
plt.axis("off")

plt.show()

In [ ]:
## Saving Visualization

OUTPUT_PATH = r"Outputs"

os.makedirs(OUTPUT_PATH, exist_ok=True)

plt.figure(figsize=(18,6))

plt.subplot(1,3,1)
plt.imshow(rgb_image)
plt.title("Original")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(heatmap_color)
plt.title("Grad-CAM")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(overlay)
plt.title("Overlay")
plt.axis("off")

plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_PATH, "GradCAM_Visualization.png"), dpi=300, bbox_inches="tight")
plt.close()

print("Visualization Saved Successfully.")

In [ ]:
## Removing Hooks

forward_handle.remove()
backward_handle.remove()

print("Hooks Removed Successfully.")